# Dunnhumby seed 43 — N/V M2 + 수정 원형 M4(λ=0.25) 결합 M5

M1, λ=0.5 M4/M5, λ=0.25 M4는 정확한 이전 결과와 checkpoint를 검증해 재사용합니다. 새 GPU 학습은 **λ=0.25 M5 한 개**뿐입니다. N/V 이력 표현은 추천 점수와 BPR 손실에 연결되어 M4 행 가중과 하나의 optimizer에서 함께 학습됩니다. 동결·사후 보정·외부 재정렬은 없습니다.

신규 상품 추천 개발분할, seed 43, binary graph, MIN_ITEM_INTER=1, 균등 미관측 음성 1개, ID 64차원·2층·L2 1e-3·ρ=0.05를 유지합니다. 최대 300 epoch, 25 epoch마다 개발평가, 전체 가격·구매금액 가중 적중값@10 최고 checkpoint 선택, 100 이후 4회 미개선 시 종료합니다. 기존 선택규칙을 바꾸지 않습니다.

M5를 동일 seed의 M1·λ=0.25 M4·λ=0.5 M5와 비교합니다. 주 판독은 전체 가격·구매금액 가중 적중값@10과 가중 NDCG@10이 M1 및 λ=0.25 M4보다 모두 높고, 전체 Recall/NDCG @10·@20·@50 각각 M1의 99% 이상인지입니다. 전 지표·저/중/고CLV·@20/@50·노출 지표·불리한 결과까지 원본 저장합니다. 반복 노출된 개발분할 단일 시드이므로 유의성·일반화·CLV 귀속을 주장하지 않습니다. test/holdout은 만들지 않습니다.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
from google.colab import drive
ROOT = Path('/content/drive/MyDrive/논문/data')
REPORT = ROOT/'results_v3_dunnhumby_original_m4_lambda025_seed43_v1/reports/result.json'
if REPORT.is_file():
    print('정확한 λ=0.25 결과가 보입니다. Drive를 재마운트하지 않습니다.')
else:
    if os.path.ismount('/content/drive'):
        raise RuntimeError('Drive는 연결됐지만 λ=0.25 결과가 없습니다. 계정과 REPORT 경로를 확인하세요. 학습은 시작되지 않았습니다.')
    try:
        drive.mount('/content/drive')
    except (ValueError, NotImplementedError) as exc:
        raise RuntimeError('Drive 연결 실패. 학습은 시작되지 않았습니다. 연결 계정을 확인하세요.') from exc
    if not REPORT.is_file():
        raise RuntimeError('Drive 연결 후에도 정확한 λ=0.25 결과가 보이지 않습니다. 학습은 시작되지 않았습니다.')
SOURCE_COMMIT = 'dcca25015e0481133b65a26abfe53d81ce160bb3'
REPO = Path('/content/clv-m5-lambda025-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
if 'lightgcn_clv_v3' in sys.modules:
    assert Path(sys.modules['lightgcn_clv_v3'].__file__).resolve().parent == REPO.resolve(), '다른 소스가 로드됐습니다. 런타임을 다시 시작하세요.'
os.chdir(REPO)
sys.path.insert(0, str(REPO))
import clv_m5_linear_nv_original_m4_lambda025_screen as screen
import pandas as pd
OUT = ROOT/'results_v3_dunnhumby_m5_linear_nv_original_m4_lambda025_seed43_v1'
print('코드 버전:', screen.VERSION, '| 새 모델:', screen.MODEL_ID)

## 1. 학습 전 출처·가중치 검증
λ=0.25 결과 JSON의 SHA-256, M1 원본, λ=0.5 M4/M5 및 λ=0.25 M4의 선택 checkpoint, 입력 해시, 무효 입력 가중치를 검증합니다. 하나라도 어긋나면 GPU 학습 전에 중단합니다.

In [ ]:
cfg, prepared, audit = screen.prepare(REPORT, OUT)
assert cfg.positive_weight_lambda == 0.25 and cfg.m4_mode == 'original'
assert cfg.seeds == (43,) and cfg.epochs == 300
assert prepared['m4_diagnostics']['original_invalid_extra_absent']
print(json.dumps({'dataset': cfg.dataset, 'seed': cfg.seeds, 'new_arm': screen.MODEL_ID, 'lambda': cfg.positive_weight_lambda, 'rho': screen.spec()['rho'], 'max_epochs': cfg.epochs, 'reused_arms': [a['model_id'] for a in prepared['anchors']], 'final_test': False, 'holdout': False}, ensure_ascii=False, indent=2))
print(audit[audit.group.isin(['all', 'any_input_invalid'])].to_string(index=False))

## 2. M5 한 개만 학습
중단되면 같은 노트북을 다시 실행하세요. 완료 epoch마다 optimizer·난수상태를 Drive에 저장하고 이어서 학습합니다. M1·M4·이전 M5는 재학습하지 않습니다.

In [ ]:
import torch
assert torch.cuda.is_available(), '학습에는 GPU 런타임을 사용하세요.'
paths = screen.run(cfg, prepared)
print(json.dumps(paths, ensure_ascii=False, indent=2))

## 3. 전체 판독과 원본 다운로드
아래 표는 주요 지표의 미리보기입니다. ZIP에는 전체·저/중/고CLV 절대지표, 모든 비교, 곡선, 판정 JSON과 가중치 감사가 들어 있습니다. 불리한 지표를 제외하지 않습니다.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
absolute = pd.read_csv(paths['absolute'])
comparison = pd.read_csv(paths['comparison'])
report = json.loads(Path(paths['json']).read_text())
key_metrics = ['recall@10', 'ndcg@10', 'recall@20', 'ndcg@20', 'recall@50', 'ndcg@50', 'price_purchase_amount_weighted_hit@10', 'vndcg@10', 'price_purchase_amount_weighted_hit@20', 'vndcg@20', 'price_purchase_amount_weighted_hit@50', 'vndcg@50', 'coverage@10', 'coverage@20', 'coverage@50', 'user_value_tendency_recommended_price_alignment']
print(absolute[['model_id', 'selected_epoch', 'stopped_epoch'] + key_metrics].set_index('model_id').T.to_string())
print('사전 판독:', json.dumps(report['reading'], ensure_ascii=False, indent=2))
print(comparison[comparison.metric.isin(key_metrics)].to_string(index=False))
zip_path = Path('/content/m5_linear_nv_original_m4_lambda025_seed43_results.zip')
with ZipFile(zip_path, 'w', compression=ZIP_DEFLATED) as archive:
    for name, path in paths.items():
        archive.write(path, arcname=Path(path).name)
    for name in ('m4_validity_audit_lambda025.csv', 'm4_validity_audit_lambda025.json'):
        archive.write(OUT/name, arcname=name)
files.download(str(zip_path))